In [2]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
# net[0] = Linear(4,8)
# net[1] = ReLU
# net[2] = Linear(8,1)
X = torch.rand(size=(2, 4))
net(X)

tensor([[0.2967],
        [0.3114]], grad_fn=<AddmmBackward0>)

In [3]:
print(net[2].state_dict())

OrderedDict({'weight': tensor([[ 0.3512, -0.3062,  0.0312,  0.3205, -0.2468,  0.2403, -0.0638,  0.1966]]), 'bias': tensor([0.3472])})


**target parameters**

In [5]:
print(type(net[2].bias))

<class 'torch.nn.parameter.Parameter'>


In [6]:
print(net[2].bias)

Parameter containing:
tensor([0.3472], requires_grad=True)


In [7]:
print(net[2].bias.data)

tensor([0.3472])


In [10]:
net[2].weight.grad == None
# right now have not performed anything so grad is nonexistent. but after ward will be seomthig

True

**Accessing all parameters**

In [11]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [12]:
# can use the string names to access the data
net.state_dict()['2.bias'].data

tensor([0.3472])

**obtaining parameters from layers**

In [13]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f"block {i}", block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)        

tensor([[0.1499],
        [0.1499]], grad_fn=<AddmmBackward0>)

In [16]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


**initialize**

In [19]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
# apply function applys the input function, init_normal, to every module in the net
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0020, -0.0066,  0.0041,  0.0086]), tensor(0.))

In [21]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[1]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [24]:
def my_init(m):
    if type(m) == nn.Linear:
        print("init", *[(name, param.shape) for name, param in m.named_parameters()][0])
        # * unpacks items, strips away the parenthesis
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5
        # x = x * y (in place multiplication)

net.apply(my_init)
net[0].weight[:2]

init weight torch.Size([8, 4])
init weight torch.Size([1, 8])


tensor([[-0.0000,  9.0829, -9.4288, -6.7063],
        [-0.0000, -0.0000, -5.0275, -6.6034]], grad_fn=<SliceBackward0>)

In [27]:
# can also directly change the parameters
print(net[0].weight.data[0])
net[0].weight.data[:] += 1
net[0].weight.data[0,0] = 42
net[0].weight.data[0]

tensor([42.0000, 11.0829, -7.4288, -4.7063])


tensor([42.0000, 12.0829, -6.4288, -3.7063])

**Sharing parameters between layers - binding parameters**

In [28]:
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), shared, nn.ReLU(), shared, nn.ReLU(), nn.Linear(8, 1))
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
